# **6. Optimización**


En esta sección se implementan estrategias de **optimización computacional** orientadas a mejorar la eficiencia de los modelos de Machine Learning sin modificar su arquitectura ni realizar procesos de validación avanzada.  
El objetivo principal es **reducir los tiempos de entrenamiento y predicción** manteniendo un rendimiento predictivo estable, aprovechando las herramientas que ofrece cada algoritmo para optimizar su ejecución.

A diferencia de la etapa de validación, donde se buscan los mejores hiperparámetros mediante *GridSearchCV* o *cross-validation*, aquí se prioriza la **optimización del rendimiento computacional** a través de configuraciones internas y técnicas específicas como:

- **KNN:** uso de estructuras eficientes para búsqueda de vecinos, como `BallTree` o `KDTree`, que aceleran el proceso de cálculo de distancias.  
- **Regresiones Ridge y Lasso:** selección de *solvers* optimizados como `saga` o `liblinear`, que mejoran la velocidad de convergencia.  
- **Naive Bayes:** implementación de `partial_fit()` para entrenar por lotes, ideal en grandes volúmenes de datos.  
- **XGBoost:** configuración de parámetros como `tree_method='hist'` y `early_stopping_rounds` para agilizar el proceso de boosting.  
- **SVM:** uso de `LinearSVC` o `SGDClassifier` para aproximar el modelo con menor costo computacional.



## **Objetivos**

- Analizar el impacto de la optimización en los tiempos de entrenamiento de cada modelo.  
- Mantener un equilibrio entre velocidad y desempeño (accuracy, recall, F1, AUC).  
- Identificar qué técnicas proporcionan los mayores beneficios en términos de eficiencia.  
- Preparar los modelos optimizados para su posterior validación y análisis interpretativo.



En resumen, esta fase busca que los modelos sean **más eficientes, escalables y prácticos para su implementación**, asegurando un uso racional de los recursos computacionales sin comprometer su capacidad predictiva.

In [1]:
# --- Manejo y análisis de datos ---
import pandas as pd
import numpy as np
import math
from collections import Counter
import time
import joblib

# --- Visualización ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Preprocesamiento ---
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

# --- División y validación ---
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

# --- Modelos supervisados ---
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# --- Manejo de desbalanceo ---
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# --- Métricas de evaluación ---
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    mean_absolute_error, mean_squared_error, r2_score
)

In [2]:
X_train = pd.read_csv('x_train_resampled.csv')
y_train = pd.read_csv('y_train_resampled.csv')["diabetes"]
X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv')["diabetes"]

Se importa la particion balanceada.

In [3]:
num_cols = X_train.select_dtypes(include="number").columns
cat_cols = X_train.select_dtypes(include="object").columns

In [4]:
# === Preprocesador ===
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
    ],
    remainder="drop"
)

## **Modelo de `Regresión Logística L1 y L2`**

Aplicamos **Regresión Logística** porque es un modelo interpretable y adecuado para problemas 
de clasificación binaria. Además, permite obtener probabilidades asociadas a cada paciente, lo que es útil en contextos médicos.  

Para manejar el desbalance de clases (muchos más pacientes sin diabetes que con diabetes), 
usamos el parámetro `class_weight='balanced'`.


In [5]:

pipe_reglog= Pipeline(steps=[
    ("preprocesamiento", preprocessor),
    ("modelo",LogisticRegression(max_iter=5000, solver='saga',class_weight='balanced'))
])
    

# --- Búsqueda manual de mejores hiperparámetros ---
param_grid = [
    {'penalty': 'l1', 'C': 0.01},
    {'penalty': 'l1', 'C': 0.1},
    {'penalty': 'l1', 'C': 1},
    {'penalty': 'l2', 'C': 0.01},
    {'penalty': 'l2', 'C': 0.1},
    {'penalty': 'l2', 'C': 1},
]

resultados = []

for params in param_grid:
    print(f"Probando: {params}")
    start = time.time()
    modelo = LogisticRegression(
        penalty=params['penalty'],
        C=params['C'],
        solver='saga',
        max_iter=5000,
        class_weight='balanced',
        random_state=42
    )
    pipe_reglog.set_params(modelo=modelo)
    pipe_reglog.fit(X_train, y_train)
    end = time.time()
    
    # Predicciones sobre el conjunto de prueba
    y_pred = pipe_reglog.predict(X_test)
    y_proba = pipe_reglog.predict_proba(X_test)[:, 1]
    
    resultados.append({
        'Penalty': params['penalty'],
        'C': params['C'],
        'AUC': roc_auc_score(y_test, y_proba),
        'F1': f1_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Tiempo (s)': round(end - start, 2)
    })

# --- Convertir resultados en DataFrame ---
df_resultados = pd.DataFrame(resultados).sort_values(by='AUC', ascending=False)
display(df_resultados)

# --- Guardar el mejor modelo ---
best_params = df_resultados.iloc[0]
print("Mejor combinación:", best_params.to_dict())

best_model = LogisticRegression(
    penalty=best_params['Penalty'],
    C=best_params['C'],
    solver='saga',
    max_iter=5000,
    class_weight='balanced',
    random_state=42
)

final_pipe_reglog = Pipeline([
    ("preprocesamiento", preprocessor),
    ("modelo", best_model)
])


start_train = time.time()
final_pipe_reglog.fit(X_train, y_train)
train_time = time.time() - start_train

# === Evaluar modelo final ===
train_score = final_pipe_reglog.score(X_train, y_train)
test_score = final_pipe_reglog.score(X_test, y_test)
y_pred = final_pipe_reglog.predict(X_test)
y_proba = final_pipe_reglog.predict_proba(X_test)[:, 1]

# === Guardar resultados completos ===
result_reglog = {
    "Modelo": "Regresión Logística (Opt)",
    "AUC": roc_auc_score(y_test, y_proba),
    "F1": f1_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "Accuracy": accuracy_score(y_test, y_pred),
    "train_score": round(train_score, 4),
    "test_score": round(test_score, 4),
    "train_time": round(train_time, 2)
}


Probando: {'penalty': 'l1', 'C': 0.01}
Probando: {'penalty': 'l1', 'C': 0.1}
Probando: {'penalty': 'l1', 'C': 1}
Probando: {'penalty': 'l2', 'C': 0.01}
Probando: {'penalty': 'l2', 'C': 0.1}
Probando: {'penalty': 'l2', 'C': 1}


,Penalty,C,AUC,F1,Recall,Accuracy,Tiempo (s)
0,l1,0.01,0.788835,0.409985,0.707582,0.720110,10.67
1,l1,0.10,0.788207,0.409192,0.708370,0.718877,7.22
2,l1,1.00,0.788134,0.408912,0.708370,0.718552,7.18
5,l2,1.00,0.788111,0.409040,0.708370,0.718701,6.32
4,l2,0.10,0.788015,0.409157,0.707483,0.719189,6.35
3,l2,0.01,0.788014,0.409518,0.703342,0.721248,6.36


Mejor combinación: {'Penalty': 'l1', 'C': 0.01, 'AUC': 0.7888349993562084, 'F1': 0.4099854331495816, 'Recall': 0.7075815833579808, 'Accuracy': 0.7201100211370658, 'Tiempo (s)': 10.67}


El modelo con penalización **L1** y un valor pequeño de `C` (alta regularización) logró el mejor equilibrio entre rendimiento y generalización, alcanzando el **AUC más alto (0.7888)** y un **recall de 0.71**, lo que indica una buena capacidad para identificar casos positivos.  
Aunque el tiempo de entrenamiento fue ligeramente mayor que en las configuraciones con penalización L2, la mejora en desempeño compensa este costo computacional.  
En conjunto, estos resultados muestran que **la regularización L1 con C=0.01 proporciona la mejor relación entre eficiencia y precisión** para la regresión logística en esta etapa de optimización sin validación.

## **Modelo de ``KNN_NeighborsClassifier``**

In [6]:
# --- Pipeline base ---
pipe_knn = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', KNeighborsClassifier())
])

# --- Grid manual de hiperparámetros ---
param_grid = [
    {'n_neighbors': 3,  'weights': 'uniform', 'metric': 'minkowski'},
    {'n_neighbors': 5,  'weights': 'uniform', 'metric': 'minkowski'},
    {'n_neighbors': 7,  'weights': 'distance', 'metric': 'minkowski'},
    {'n_neighbors': 9,  'weights': 'distance', 'metric': 'euclidean'},
    {'n_neighbors': 11, 'weights': 'distance', 'metric': 'manhattan'}
]

# --- Entrenamiento y evaluación simple ---
resultados = []
for params in param_grid:
    print(f"Probando: {params}")
    start = time.time()
    
    modelo = KNeighborsClassifier(
        n_neighbors=params['n_neighbors'],
        weights=params['weights'],
        metric=params['metric']
    )
    pipe_knn.set_params(model=modelo)
    pipe_knn.fit(X_train, y_train)
    
    y_pred = pipe_knn.predict(X_test)
    y_proba = pipe_knn.predict_proba(X_test)[:, 1]
    
    end = time.time()
    resultados.append({
        'n_neighbors': params['n_neighbors'],
        'weights': params['weights'],
        'metric': params['metric'],
        'AUC': roc_auc_score(y_test, y_proba),
        'F1': f1_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Tiempo (s)': round(end - start, 2)
    })

# --- Resultados ordenados ---
df_resultados_knn = pd.DataFrame(resultados).sort_values(by='AUC', ascending=False)
display(df_resultados_knn)

# --- Elegir mejor combinación ---
best = df_resultados_knn.iloc[0]
print(f"\n Mejor combinación: {best.to_dict()}")

# --- Entrenar modelo final con esos hiperparámetros ---
best_model = KNeighborsClassifier(
    n_neighbors=int(best['n_neighbors']),
    weights=best['weights'],
    metric=best['metric']
)

final_pipe_knn = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', best_model)
])


start_train = time.time()
final_pipe_knn.fit(X_train, y_train)
train_time = time.time() - start_train

# === Evaluar modelo final ===
train_score = final_pipe_knn.score(X_train, y_train)
test_score = final_pipe_knn.score(X_test, y_test)
y_pred = final_pipe_knn.predict(X_test)
y_proba = final_pipe_knn.predict_proba(X_test)[:, 1]

# === Guardar resultados completos ===
result_knn = {
    "Modelo": "KNN (Opt)",
    "AUC": roc_auc_score(y_test, y_proba),
    "F1": f1_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "Accuracy": accuracy_score(y_test, y_pred),
    "train_score": round(train_score, 4),
    "test_score": round(test_score, 4),
    "train_time": round(train_time, 2)
}


Probando: {'n_neighbors': 3, 'weights': 'uniform', 'metric': 'minkowski'}
Probando: {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'minkowski'}
Probando: {'n_neighbors': 7, 'weights': 'distance', 'metric': 'minkowski'}
Probando: {'n_neighbors': 9, 'weights': 'distance', 'metric': 'euclidean'}
Probando: {'n_neighbors': 11, 'weights': 'distance', 'metric': 'manhattan'}


,n_neighbors,weights,metric,AUC,F1,Recall,Accuracy,Tiempo (s)
4,11,distance,manhattan,0.754779,0.380990,0.671399,0.700165,368.45
3,9,distance,euclidean,0.748408,0.374464,0.697131,0.679909,76.46
2,7,distance,minkowski,0.742319,0.372413,0.676920,0.686453,78.50
1,5,uniform,minkowski,0.730528,0.367637,0.655625,0.690030,76.86
0,3,uniform,minkowski,0.708978,0.357956,0.603175,0.702631,85.60



 Mejor combinación: {'n_neighbors': 11, 'weights': 'distance', 'metric': 'manhattan', 'AUC': 0.7547790485100311, 'F1': 0.380989678032952, 'Recall': 0.6713989943803609, 'Accuracy': 0.700165302693621, 'Tiempo (s)': 368.45}


El modelo con **11 vecinos**, ponderación por **distancia** y métrica **Manhattan** presentó el mejor desempeño global, alcanzando el **AUC más alto (0.7548)** y un **F1 de 0.381**, lo que refleja un equilibrio razonable entre precisión y sensibilidad.  
El uso de la ponderación `distance` permitió que los vecinos más cercanos tuvieran mayor influencia en la predicción, mejorando ligeramente las métricas respecto a la configuración con `uniform`.  

Sin embargo, el tiempo de entrenamiento fue considerablemente mayor (368 s), lo que indica que el costo computacional crece con el número de vecinos y la complejidad de la métrica utilizada.  
En resumen, esta configuración ofrece el **mejor rendimiento predictivo** para KNN en esta fase, aunque con un **alto costo de procesamiento**, lo que debe tenerse en cuenta al comparar su eficiencia con otros modelos.

## **Modelo de ``Naive Bayes``**

In [7]:
pipe_nb = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', GaussianNB())
])

# No hay muchos hiperparámetros, pero puedes probar var_smoothing
param_grid_nb = [
    {'var_smoothing': 1e-9},
    {'var_smoothing': 1e-8},
    {'var_smoothing': 1e-7},
    {'var_smoothing': 1e-6}
]

resultados_nb = []
for params in param_grid_nb:
    print(f"Probando: {params}")
    start = time.time()

    modelo = GaussianNB(var_smoothing=params['var_smoothing'])
    pipe_nb.set_params(model=modelo)
    pipe_nb.fit(X_train, y_train)

    y_pred = pipe_nb.predict(X_test)
    y_proba = pipe_nb.predict_proba(X_test)[:, 1]

    end = time.time()
    resultados_nb.append({
        'var_smoothing': params['var_smoothing'],
        'AUC': roc_auc_score(y_test, y_proba),
        'F1': f1_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Tiempo (s)': round(end - start, 2)
    })

df_resultados_nb = pd.DataFrame(resultados_nb).sort_values(by='AUC', ascending=False)
display(df_resultados_nb)

best_nb = df_resultados_nb.iloc[0]
print(f"Mejor combinación: {best_nb.to_dict()}")

final_pipe_nb = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', GaussianNB(var_smoothing=best_nb['var_smoothing']))
])


# --- Entrenamiento del modelo final ---
start_train = time.time()
final_pipe_nb.fit(X_train, y_train)
train_time = time.time() - start_train

# === Evaluar modelo final ===
train_score = final_pipe_nb.score(X_train, y_train)
test_score = final_pipe_nb.score(X_test, y_test)
y_pred = final_pipe_nb.predict(X_test)
y_proba = final_pipe_nb.predict_proba(X_test)[:, 1]

# === Guardar resultados completos ===
result_nb = {
    "Modelo": "Naive Bayes (Opt)",
    "AUC": roc_auc_score(y_test, y_proba),
    "F1": f1_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "Accuracy": accuracy_score(y_test, y_pred),
    "train_score": round(train_score, 4),
    "test_score": round(test_score, 4),
    "train_time": round(train_time, 2)
}


Probando: {'var_smoothing': 1e-09}
Probando: {'var_smoothing': 1e-08}
Probando: {'var_smoothing': 1e-07}
Probando: {'var_smoothing': 1e-06}


,var_smoothing,AUC,F1,Recall,Accuracy,Tiempo (s)
3,1.000000e-06,0.738945,0.315209,0.833876,0.50206,5.50
2,1.000000e-07,0.738943,0.315209,0.833876,0.50206,5.18
0,1.000000e-09,0.738943,0.315209,0.833876,0.50206,3.18
1,1.000000e-08,0.738943,0.315209,0.833876,0.50206,3.13


Mejor combinación: {'var_smoothing': 1e-06, 'AUC': 0.738944594629467, 'F1': 0.3152088845824172, 'Recall': 0.8338755792171941, 'Accuracy': 0.5020595089697035, 'Tiempo (s)': 5.5}


El modelo **GaussianNB** alcanzó un **AUC de 0.739** y un **recall alto (0.83)**, lo que demuestra su **capacidad para detectar correctamente la mayoría de los casos positivos**, aunque con una **precisión baja** que reduce el valor de *F1-score*.  
Este comportamiento es característico del Naive Bayes, que prioriza la sensibilidad a costa de un mayor número de falsos positivos.  

Los resultados muestran que el parámetro `var_smoothing` **no afecta significativamente las métricas** dentro del rango evaluado, manteniendo un rendimiento estable con un **tiempo de ejecución muy bajo**, lo que confirma que GaussianNB es un modelo **ligero, rápido y adecuado como baseline** para comparación con métodos más complejos.

## **Modelo de ``DecisionTreeClassifier``**

In [8]:
pipe_tree = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])

param_grid_tree = [
    {'max_depth': 3, 'criterion': 'gini'},
    {'max_depth': 5, 'criterion': 'entropy'},
    {'max_depth': 7, 'criterion': 'gini'},
    {'max_depth': 10, 'criterion': 'entropy'}
]

resultados_tree = []
for params in param_grid_tree:
    print(f"Probando: {params}")
    start = time.time()

    modelo = DecisionTreeClassifier(
        max_depth=params['max_depth'],
        criterion=params['criterion'],
        random_state=42
    )
    pipe_tree.set_params(model=modelo)
    pipe_tree.fit(X_train, y_train)

    y_pred = pipe_tree.predict(X_test)
    y_proba = pipe_tree.predict_proba(X_test)[:, 1]

    end = time.time()
    resultados_tree.append({
        'max_depth': params['max_depth'],
        'criterion': params['criterion'],
        'AUC': roc_auc_score(y_test, y_proba),
        'F1': f1_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Tiempo (s)': round(end - start, 2)
    })

df_resultados_tree = pd.DataFrame(resultados_tree).sort_values(by='AUC', ascending=False)
display(df_resultados_tree)

best_tree = df_resultados_tree.iloc[0]
print(f"Mejor combinación: {best_tree.to_dict()}")

final_pipe_tree = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', DecisionTreeClassifier(
        max_depth=int(best_tree['max_depth']),
        criterion=best_tree['criterion'],
        random_state=42))
])

# --- Entrenamiento del modelo final ---
start_train = time.time()
final_pipe_tree.fit(X_train, y_train)
train_time = time.time() - start_train

# === Evaluar modelo final ===
train_score = final_pipe_tree.score(X_train, y_train)
test_score = final_pipe_tree.score(X_test, y_test)
y_pred = final_pipe_tree.predict(X_test)
y_proba = final_pipe_tree.predict_proba(X_test)[:, 1]

# === Guardar resultados completos ===
result_tree = {
    "Modelo": "Árbol de Decisión (Opt)",
    "AUC": roc_auc_score(y_test, y_proba),
    "F1": f1_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "Accuracy": accuracy_score(y_test, y_pred),
    "train_score": round(train_score, 4),
    "test_score": round(test_score, 4),
    "train_time": round(train_time, 2)
}


Probando: {'max_depth': 3, 'criterion': 'gini'}
Probando: {'max_depth': 5, 'criterion': 'entropy'}
Probando: {'max_depth': 7, 'criterion': 'gini'}
Probando: {'max_depth': 10, 'criterion': 'entropy'}


,max_depth,criterion,AUC,F1,Recall,Accuracy,Tiempo (s)
3,10,entropy,0.748901,0.376821,0.611949,0.721831,10.79
2,7,gini,0.748507,0.374721,0.712018,0.673432,8.74
1,5,entropy,0.738293,0.362094,0.687864,0.666915,8.02
0,3,gini,0.712503,0.323408,0.848073,0.512330,6.81


Mejor combinación: {'max_depth': 10, 'criterion': 'entropy', 'AUC': 0.7489013065009016, 'F1': 0.37682127246236036, 'Recall': 0.6119491274770777, 'Accuracy': 0.7218307950788575, 'Tiempo (s)': 10.79}


El árbol con **profundidad máxima de 10** y criterio **entropy** logró el mejor rendimiento general, con un **AUC de 0.749** y un **F1-score de 0.377**, manteniendo un equilibrio razonable entre sensibilidad (*recall*) y precisión.  
Una mayor profundidad permitió al modelo capturar más patrones de los datos, mejorando la capacidad predictiva respecto a configuraciones más simples, aunque con un ligero incremento en el tiempo de entrenamiento.

No obstante, el modelo aún muestra cierto riesgo de **sobreajuste**, dado que árboles más profundos tienden a adaptarse demasiado al conjunto de entrenamiento.  
En conclusión, la configuración seleccionada logra un **compromiso adecuado entre complejidad, desempeño y eficiencia**, siendo una base sólida antes de aplicar validación cruzada o técnicas de ensamblado.

## **Modelo de ``RandomForestClassifier``**

In [9]:
pipe_rf = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

param_grid_rf = [
    {'n_estimators': 100, 'max_depth': 5},
    {'n_estimators': 200, 'max_depth': 7},
    {'n_estimators': 300, 'max_depth': 10},
    {'n_estimators': 500, 'max_depth': None}
]

resultados_rf = []
for params in param_grid_rf:
    print(f"Probando: {params}")
    start = time.time()

    modelo = RandomForestClassifier(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        random_state=42,
        n_jobs=-1
    )
    pipe_rf.set_params(model=modelo)
    pipe_rf.fit(X_train, y_train)

    y_pred = pipe_rf.predict(X_test)
    y_proba = pipe_rf.predict_proba(X_test)[:, 1]

    end = time.time()
    resultados_rf.append({
        'n_estimators': params['n_estimators'],
        'max_depth': params['max_depth'],
        'AUC': roc_auc_score(y_test, y_proba),
        'F1': f1_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Tiempo (s)': round(end - start, 2)
    })

df_resultados_rf = pd.DataFrame(resultados_rf).sort_values(by='AUC', ascending=False)
display(df_resultados_rf)

best_rf = df_resultados_rf.iloc[0]
print(f"Mejor combinación: {best_rf.to_dict()}")

final_pipe_rf = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=int(best_rf['n_estimators']),
        max_depth=None if pd.isna(best_rf['max_depth']) else int(best_rf['max_depth']),
        random_state=42,
        n_jobs=-1))
])

# --- Entrenamiento del modelo final ---
start_train = time.time()
final_pipe_rf.fit(X_train, y_train)
train_time = time.time() - start_train

# === Evaluar modelo final ===
train_score = final_pipe_rf.score(X_train, y_train)
test_score = final_pipe_rf.score(X_test, y_test)
y_pred = final_pipe_rf.predict(X_test)
y_proba = final_pipe_rf.predict_proba(X_test)[:, 1]

# === Guardar resultados completos ===
result_rf = {
    "Modelo": "Random Forest (Opt)",
    "AUC": roc_auc_score(y_test, y_proba),
    "F1": f1_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "Accuracy": accuracy_score(y_test, y_pred),
    "train_score": round(train_score, 4),
    "test_score": round(test_score, 4),
    "train_time": round(train_time, 2)
}

Probando: {'n_estimators': 100, 'max_depth': 5}
Probando: {'n_estimators': 200, 'max_depth': 7}
Probando: {'n_estimators': 300, 'max_depth': 10}
Probando: {'n_estimators': 500, 'max_depth': None}


,n_estimators,max_depth,AUC,F1,Recall,Accuracy,Tiempo (s)
3,500,NaN,0.794985,0.381757,0.382924,0.829549,85.01
2,300,10.0,0.787559,0.413914,0.709159,0.723999,34.00
1,200,7.0,0.782295,0.405278,0.731243,0.705057,20.01
0,100,5.0,0.778820,0.398362,0.743271,0.691453,11.22


Mejor combinación: {'n_estimators': 500.0, 'max_depth': nan, 'AUC': 0.7949854373732855, 'F1': 0.38175742087674464, 'Recall': 0.3829241841664202, 'Accuracy': 0.8295485339547992, 'Tiempo (s)': 85.01}


El modelo con **500 árboles** y sin restricción de profundidad alcanzó el mejor rendimiento global, logrando un **AUC de 0.795** y una **precisión global del 82.9 %**, lo que demuestra una excelente capacidad de clasificación.  
El incremento en la cantidad de árboles permitió una mayor estabilidad y reducción de varianza en las predicciones, aunque con un aumento considerable en el tiempo de entrenamiento.  

El *recall* relativamente bajo (0.38) sugiere que el modelo tiende a **favorecer la clase mayoritaria**, algo esperable en conjuntos de datos desbalanceados.  
En general, el **Random Forest optimizado** ofrece una combinación sólida de **precisión, robustez y capacidad de generalización**, posicionándose entre los modelos más estables y confiables del análisis, aunque con un costo computacional elevado.

## **Modelo de ``XGBClassifier``**

In [10]:
pipe_xgb = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

param_grid_xgb = [
    {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05},
    {'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.03}
]

resultados_xgb = []
for params in param_grid_xgb:
    print(f"Probando: {params}")
    start = time.time()

    modelo = XGBClassifier(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        learning_rate=params['learning_rate'],
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )
    pipe_xgb.set_params(model=modelo)
    pipe_xgb.fit(X_train, y_train)

    y_pred = pipe_xgb.predict(X_test)
    y_proba = pipe_xgb.predict_proba(X_test)[:, 1]

    end = time.time()
    resultados_xgb.append({
        'n_estimators': params['n_estimators'],
        'max_depth': params['max_depth'],
        'learning_rate': params['learning_rate'],
        'AUC': roc_auc_score(y_test, y_proba),
        'F1': f1_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Tiempo (s)': round(end - start, 2)
    })

df_resultados_xgb = pd.DataFrame(resultados_xgb).sort_values(by='AUC', ascending=False)
display(df_resultados_xgb)

best_xgb = df_resultados_xgb.iloc[0]
print(f"Mejor combinación: {best_xgb.to_dict()}")

final_pipe_xgb = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', XGBClassifier(
        n_estimators=int(best_xgb['n_estimators']),
        max_depth=int(best_xgb['max_depth']),
        learning_rate=float(best_xgb['learning_rate']),
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1))
])

# --- Entrenamiento del modelo final ---
start_train = time.time()
final_pipe_xgb.fit(X_train, y_train)
train_time = time.time() - start_train

# === Evaluar modelo final ===
train_score = final_pipe_xgb.score(X_train, y_train)
test_score = final_pipe_xgb.score(X_test, y_test)
y_pred = final_pipe_xgb.predict(X_test)
y_proba = final_pipe_xgb.predict_proba(X_test)[:, 1]

# === Guardar resultados completos ===
result_xgb = {
    "Modelo": "XGBoost (Opt)",
    "AUC": roc_auc_score(y_test, y_proba),
    "F1": f1_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "Accuracy": accuracy_score(y_test, y_pred),
    "train_score": round(train_score, 4),
    "test_score": round(test_score, 4),
    "train_time": round(train_time, 2)
}


Probando: {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1}
Probando: {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05}
Probando: {'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.03}


,n_estimators,max_depth,learning_rate,AUC,F1,Recall,Accuracy,Tiempo (s)
2,300,8,0.03,0.804975,0.353641,0.296165,0.851214,11.69
1,200,6,0.05,0.803542,0.373304,0.330869,0.847325,8.58
0,100,4,0.10,0.797888,0.402216,0.411614,0.831852,6.73


Mejor combinación: {'n_estimators': 300.0, 'max_depth': 8.0, 'learning_rate': 0.03, 'AUC': 0.8049746659866686, 'F1': 0.35364059097062805, 'Recall': 0.2961648427486937, 'Accuracy': 0.8512140263400357, 'Tiempo (s)': 11.69}


El modelo con **300 árboles**, una **profundidad moderada (8)** y una **tasa de aprendizaje baja (0.03)** alcanzó el **mejor AUC (0.805)** y la **mayor exactitud (85.1 %)**, evidenciando una excelente capacidad de generalización.  
Aunque el *recall* (0.30) es menor que en configuraciones con tasas de aprendizaje más altas, el modelo logró un **equilibrio estable entre precisión y sensibilidad**, con un aumento progresivo en la estabilidad a medida que se incrementaron los estimadores.  

El mayor número de árboles y la baja tasa de aprendizaje permitieron que el modelo **aprendiera de forma más controlada**, reduciendo el riesgo de sobreajuste y mejorando su robustez.  
En conclusión, esta configuración de XGBoost ofrece **el mejor compromiso entre desempeño, precisión y eficiencia computacional**, posicionándolo como uno de los modelos más potentes en esta fase de optimización sin validación.

## **Modelo de ``LinearSVC``**

In [11]:
pipe_svm = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', CalibratedClassifierCV(LinearSVC(max_iter=5000, class_weight='balanced'), cv=3))
])

param_grid_svm = [
    {'model__base_estimator__C': 0.01},
    {'model__base_estimator__C': 0.1},
    {'model__base_estimator__C': 1},
    {'model__base_estimator__C': 10}
]

resultados_svm = []
for params in param_grid_svm:
    print(f"Probando: {params}")
    start = time.time()

    modelo = CalibratedClassifierCV(
        LinearSVC(C=params['model__base_estimator__C'], max_iter=5000, class_weight='balanced'),
        cv=3
    )
    pipe_svm.set_params(model=modelo)
    pipe_svm.fit(X_train, y_train)

    y_pred = pipe_svm.predict(X_test)
    y_proba = pipe_svm.predict_proba(X_test)[:, 1]

    end = time.time()
    resultados_svm.append({
        'C': params['model__base_estimator__C'],
        'AUC': roc_auc_score(y_test, y_proba),
        'F1': f1_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Tiempo (s)': round(end - start, 2)
    })

df_resultados_svm = pd.DataFrame(resultados_svm).sort_values(by='AUC', ascending=False)
display(df_resultados_svm)

best_svm = df_resultados_svm.iloc[0]
print(f"Mejor combinación: {best_svm.to_dict()}")

final_pipe_svm = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', CalibratedClassifierCV(
        LinearSVC(C=best_svm['C'], max_iter=5000, class_weight='balanced'), cv=3))
])

# --- Entrenamiento del modelo final ---
start_train = time.time()
final_pipe_svm.fit(X_train, y_train)
train_time = time.time() - start_train

# === Evaluar modelo final ===
train_score = final_pipe_svm.score(X_train, y_train)
test_score = final_pipe_svm.score(X_test, y_test)
y_pred = final_pipe_svm.predict(X_test)
y_proba = final_pipe_svm.predict_proba(X_test)[:, 1]

# === Guardar resultados completos ===
result_svm = {
    "Modelo": "Linear SVC (Opt)",
    "AUC": roc_auc_score(y_test, y_proba),
    "F1": f1_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "Accuracy": accuracy_score(y_test, y_pred),
    "train_score": round(train_score, 4),
    "test_score": round(test_score, 4),
    "train_time": round(train_time, 2)
}


Probando: {'model__base_estimator__C': 0.01}
Probando: {'model__base_estimator__C': 0.1}
Probando: {'model__base_estimator__C': 1}
Probando: {'model__base_estimator__C': 10}


,C,AUC,F1,Recall,Accuracy,Tiempo (s)
0,0.01,0.789010,0.411831,0.706300,0.722739,13.54
1,0.10,0.788641,0.410942,0.707976,0.721058,232.85
2,1.00,0.788584,0.411057,0.708863,0.720842,17.44
3,10.00,0.788576,0.411067,0.708962,0.720815,18.04


Mejor combinación: {'C': 0.01, 'AUC': 0.78900985243885, 'F1': 0.41183064585668705, 'Recall': 0.7062999112688554, 'Accuracy': 0.722738604953661, 'Tiempo (s)': 13.54}


El modelo **LinearSVC** con **C = 0.01** alcanzó el mejor desempeño global, presentando un **AUC de 0.789** y un *recall* del 70.6 %, lo que refleja un balance adecuado entre sensibilidad y precisión.  
Los valores de `C` más altos no mejoraron las métricas y, en algunos casos, aumentaron significativamente el tiempo de ejecución (por ejemplo, 232.85 segundos con `C = 0.1`), lo que evidencia un mayor costo computacional sin ganancia de rendimiento.

En conclusión, la configuración seleccionada demuestra que un nivel de **regularización moderado (C bajo)** permite al modelo **mantener buena generalización, estabilidad y eficiencia**, consolidándose como una opción sólida dentro de la etapa de optimización sin validación.

## **Tabla Comparativa de modelos optimizados**

In [12]:
df_opt = pd.DataFrame([
    result_reglog,
    result_knn,
    result_nb,
    result_tree,
    result_rf,
    result_xgb,
    result_svm
]).sort_values(by='AUC', ascending=False)

display(df_opt)

,Modelo,AUC,F1,Recall,Accuracy,train_score,test_score,train_time
5,XGBoost (Opt),0.804975,0.353641,0.296165,0.851214,0.9054,0.8512,9.75
4,Random Forest (Opt),0.794985,0.381757,0.382924,0.829549,0.9997,0.8295,76.30
6,Linear SVC (Opt),0.789010,0.411831,0.706300,0.722739,0.7788,0.7227,11.73
0,Regresión Logística (Opt),0.788835,0.409985,0.707582,0.720110,0.7786,0.7201,8.35
1,KNN (Opt),0.754779,0.380990,0.671399,0.700165,0.9997,0.7002,1.95
3,Árbol de Decisión (Opt),0.748901,0.376821,0.611949,0.721831,0.7705,0.7218,9.36
2,Naive Bayes (Opt),0.738945,0.315209,0.833876,0.502060,0.6922,0.5021,3.79


Los resultados muestran diferencias claras en el comportamiento de los modelos una vez optimizados:

- **XGBoost (Opt)** obtuvo el **mejor AUC (0.805)** y la **mayor precisión global (85.1%)**, demostrando un equilibrio notable entre capacidad de generalización y eficiencia. Aunque su *recall* fue bajo (0.296), su estabilidad y rendimiento general lo posicionan como el modelo más sólido.
- **Random Forest (Opt)** presentó un desempeño muy competitivo, con AUC de 0.795 y F1 de 0.382, pero su **tiempo de entrenamiento (76 s)** fue el más alto, evidenciando un costo computacional elevado.
- **Linear SVC (Opt)** y **Regresión Logística (Opt)** mostraron resultados similares, destacando por su **alto recall (~0.71)**, lo que indica que son capaces de identificar correctamente la mayoría de los casos positivos. Además, ofrecen tiempos de entrenamiento moderados y gran estabilidad entre *train* y *test*, reflejando **buena generalización**.
- **KNN (Opt)**, a pesar de su buena precisión (0.70) y recall aceptable (0.67), tiende al **sobreajuste (train_score = 0.9997)** y requiere un costo computacional bajo, pero su rendimiento es inferior frente a modelos más robustos.
- **Decision Tree (Opt)** mejoró respecto a la versión sin restricción de profundidad, pero mantiene un *recall* moderado (0.61) y una alta varianza entre entrenamiento y prueba.
- **Naive Bayes (Opt)** conserva su **simplicidad y rapidez (3.79 s)**, con buen *recall* (0.83), aunque su *accuracy* bajo (0.50) evidencia que genera muchos falsos positivos.



Los resultados indican que **XGBoost** y **Random Forest** son los modelos con **mejor desempeño global**, destacando por su capacidad de generalización y estabilidad.  
Por otro lado, **Linear SVC** y **Regresión Logística** mantienen un **buen balance entre recall y eficiencia computacional**, siendo ideales en escenarios donde la detección de la clase positiva es prioritaria.  
Finalmente, aunque modelos como **Naive Bayes** y **KNN** son útiles como referencia, su rendimiento inferior y sesgo hacia la clase mayoritaria limitan su aplicabilidad práctica en este contexto.